# Notebook 03 — Group-stratifizierter Train/Val/Test-Split

**Ziel:** `data/splits/{train,val,test}.csv` erzeugen.

**Strategie:**
- Disk ist Ground Truth: nur Alben mit vorhandenem Cover werden berücksichtigt.
- Split-Einheit ist der **Artist**, nicht das Album: alle Alben eines Artists
  landen im selben Split → kein Datenleck durch visuelle Ähnlichkeit.
- **Stratifizierung** pro Genre: Artists werden innerhalb jedes Genres
  70 / 15 / 15 aufgeteilt.
- **SEED = 42** für Reproduzierbarkeit.

**Invariante (assert):** Kein `artist_id` darf in mehr als einem Split erscheinen.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT       = Path.cwd().parent
COVERS_DIR = ROOT / "data" / "covers"
SPLITS_DIR = ROOT / "data" / "splits"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42

print(f"ROOT:       {ROOT}")
print(f"COVERS_DIR: {COVERS_DIR}")
print(f"SPLITS_DIR: {SPLITS_DIR}")

ROOT:       /Users/tjarek/uni/album-genre-classifier
COVERS_DIR: /Users/tjarek/uni/album-genre-classifier/data/covers
SPLITS_DIR: /Users/tjarek/uni/album-genre-classifier/data/splits


## 1. Disk-Scan — Cover als Ground Truth

In [2]:
rows = []
for jpg in sorted(COVERS_DIR.rglob("*.jpg")):
    # Dateiname: {artist_name}_{album_id}.jpg → album_id = letzter _-separierter Teil
    album_id = jpg.stem.rsplit("_", 1)[-1]
    rows.append({
        "album_id":   album_id,
        "genre":      jpg.parent.name,
        "cover_path": str(jpg.relative_to(ROOT)),  # relativ zum Repo-Root
    })

disk_df = pd.DataFrame(rows)
print(f"Cover auf Disk: {len(disk_df)}")
print(disk_df["genre"].value_counts().sort_index().to_string())

Cover auf Disk: 2549
genre
alternative_rock    253
classical           208
country             222
hiphop              235
house               274
indie_rock          263
jazz                271
metal               275
reggae              249
techno              299


## 2. Metadaten joinen (artist_id aus albums_raw.csv)

In [3]:
raw = pd.read_csv(ROOT / "data" / "albums_raw.csv")
raw = raw.drop_duplicates(subset="album_id", keep="first")

df = disk_df.merge(
    raw[["album_id", "artist_id"]],
    on="album_id",
    how="left",
)

missing_artist = df["artist_id"].isna().sum()
print(f"Alben ohne artist_id-Match: {missing_artist}")
if missing_artist > 0:
    print("WARNUNG: Diese Alben werden aus dem Split ausgeschlossen.")
    df = df.dropna(subset=["artist_id"]).reset_index(drop=True)

print(f"Alben für Split: {len(df)}")

Alben ohne artist_id-Match: 0
Alben für Split: 2549


## 3. Group-stratifizierter Split 70 / 15 / 15 nach artist_id

In [4]:
rng = np.random.default_rng(SEED)
df["split"] = None

for genre, group in df.groupby("genre"):
    artists = group["artist_id"].unique().tolist()  # list für rng.shuffle
    rng.shuffle(artists)
    n       = len(artists)
    n_train = int(0.70 * n)
    n_val   = int(0.15 * n)

    train_artists = set(artists[:n_train])
    val_artists   = set(artists[n_train : n_train + n_val])
    test_artists  = set(artists[n_train + n_val :])

    mask = df["genre"] == genre
    df.loc[mask & df["artist_id"].isin(train_artists), "split"] = "train"
    df.loc[mask & df["artist_id"].isin(val_artists),   "split"] = "val"
    df.loc[mask & df["artist_id"].isin(test_artists),  "split"] = "test"

print("\nAlben pro Split:")
print(df["split"].value_counts().to_string())
print("\nAlben pro Genre × Split:")
print(df.groupby(["genre", "split"]).size().unstack(fill_value=0).to_string())


Alben pro Split:
split
train    1777
test      413
val       359

Alben pro Genre × Split:
split             test  train  val
genre                             
alternative_rock    37    174   42
classical           22    150   36
country             45    148   29
hiphop              35    167   33
house               34    205   35
indie_rock          49    185   29
jazz                44    183   44
metal               42    193   40
reggae              59    161   29
techno              46    211   42


## 4. Invariante: kein Artist in mehr als einem Split

In [5]:
violations = 0
for artist, sub in df.groupby("artist_id"):
    splits_seen = sub["split"].unique()
    if len(splits_seen) > 1:
        print(f"VERLETZUNG: artist {artist} in {splits_seen}")
        violations += 1

assert violations == 0, f"{violations} Artist(s) in mehreren Splits — Datenleck!"
print(f"Invariante OK: 0 Verletzungen.")

Invariante OK: 0 Verletzungen.


## 5. CSVs schreiben

In [6]:
cols = ["album_id", "genre", "artist_id", "cover_path"]
for split in ["train", "val", "test"]:
    out_path = SPLITS_DIR / f"{split}.csv"
    df[df["split"] == split][cols].to_csv(out_path, index=False)
    n = len(df[df["split"] == split])
    print(f"  {split}.csv  →  {n} Alben  →  {out_path}")

print("\nFertig.")

  train.csv  →  1777 Alben  →  /Users/tjarek/uni/album-genre-classifier/data/splits/train.csv
  val.csv  →  359 Alben  →  /Users/tjarek/uni/album-genre-classifier/data/splits/val.csv
  test.csv  →  413 Alben  →  /Users/tjarek/uni/album-genre-classifier/data/splits/test.csv

Fertig.
